# Assignment 3 – Parte 1: Visualización

## 1. Pregunta y datos

**Pregunta:** ¿El Estado declara emergencia donde más llueve? ¿Cambió esa relación entre las temporadas de lluvias 2022 y 2025?

**Dataset:** `datos/tabla_final_2022_2025.csv`. Es la `tabla_final.csv` de nuestro Assignment 2 (1 de enero al 31 de mayo de 2024), pero junto con **las tres temporadas restantes** (1 de enero al 31 de mayo de 2022, 2023 y 2025). Cada fila es un departamento en una temporada: 25 departamentos × 4 temporadas = 100 filas.

**Columnas que usamos:**

| Columna | Qué mide |
|---|---|
| `temporada` | Año de la temporada de lluvias (ene–may) |
| `ubigeo` | Código INEI del departamento (texto de 2 dígitos) |
| `departamento` | Nombre del departamento |
| `lluvia_total_mm` | Lluvia acumulada en la capital del departamento durante la temporada (Open-Meteo) |
| `dias_lluvia_fuerte` | Días con 20 mm o más |
| `declaratorias` | Decretos supremos de la PCM que declaran emergencia por lluvias y mencionan al departamento |
| `prorrogas` | Decretos que prorrogan una emergencia ya declarada |

### Innovación: comparación de cuatro temporadas

En el Assignment 2 nuestro grupo trabajó **solo la temporada 2024**. Para este trabajo unimos nuestra tabla con las de las temporadas **2022, 2023 y 2025**, que trabajaron otros grupos con el mismo método. Esto permite ver si la relación entre lluvia y declaratorias se repite cada año o si cambia. La data de las temporadas 2022, 2023 y 2025 se obtuvo de los comentarios de los estudiantes del Diplomado en el issue del Assignment 2: https://github.com/alexanderquispe/Diplomado_PUCP/issues/1881. Se revisaron carpetas de distintos grupos encargados de la misma temporada para asegurar que la data fuese consistente.

In [1]:
import pandas as pd
import plotly.express as px

# ubigeo como texto para no perder el cero inicial ("01", no 1)
df = pd.read_csv("datos/tabla_final_2022_2025.csv", dtype={"ubigeo": str}, encoding="utf-8-sig")

print(df.shape)
df.head()

(100, 10)


,temporada,ubigeo,departamento,capital,latitud,longitud,lluvia_total_mm,dias_lluvia_fuerte,declaratorias,prorrogas
0,2022,01,Amazonas,Chachapoyas,-6.23169,-77.86903,436.5,1,1,0
1,2022,02,Áncash,Huaraz,-9.52614,-77.52869,975.3,7,0,0
2,2022,03,Apurímac,Abancay,-13.63426,-72.88380,711.8,2,0,0
3,2022,04,Arequipa,Arequipa,-16.39899,-71.53747,189.9,0,0,0
4,2022,05,Ayacucho,Ayacucho,-13.16380,-74.22345,485.1,0,1,0


### Verificación del dataset unido

Antes de graficar, revisamos que la unión de las cuatro temporadas esté completa: 25 departamentos por temporada, sin filas repetidas, sin valores vacíos y con el ubigeo de 2 dígitos.

In [2]:
print("Departamentos por temporada:", df.groupby("temporada")["ubigeo"].nunique().to_dict())
print("Filas repetidas (temporada + ubigeo):", df.duplicated(["temporada", "ubigeo"]).sum())
print("Valores vacíos:", df.isna().sum().sum())
print("Ubigeos con 2 dígitos:", df["ubigeo"].str.len().eq(2).all())

Departamentos por temporada: {2022: 25, 2023: 25, 2024: 25, 2025: 25}
Filas repetidas (temporada + ubigeo): 0
Valores vacíos: 0
Ubigeos con 2 dígitos: True


In [3]:
# Totales por temporada
df.groupby("temporada")[["lluvia_total_mm", "dias_lluvia_fuerte", "declaratorias", "prorrogas"]].sum()

,lluvia_total_mm,dias_lluvia_fuerte,declaratorias,prorrogas
temporada,,,,
2022,13315.5,88,3,0
2023,16091.3,151,72,6
2024,13137.4,75,99,68
2025,17709.4,156,157,71


ℹ️ **Sobre 2022:** esa temporada solo tiene **3 declaratorias** en todo el país (Amazonas, Ayacucho y Piura, una cada uno), frente a 72, 99 y 157 en las otras temporadas. Como la diferencia es tan grande, la verificamos en los repositorios de los cuatro grupos que trabajaron 2022 (grupos 1, 5, 9 y 13). Todos encontraron los mismos 29 resultados en el buscador de gob.pe y **un solo decreto de lluvias**: el Decreto Supremo N.° 032-2022-PCM (1 de abril de 2022), que menciona esos tres departamentos. Los otros 28 eran emergencias por COVID-19, sismos o seguridad. El grupo 1 además repitió la búsqueda con los términos "precipitaciones" y "lluvias" y obtuvo el mismo resultado. Así que no es un error de scraping: es lo que publica gob.pe para esa temporada.

In [4]:
# Fuente y estilo comunes para todos los gráficos (Sesión 6, sección 7)
FUENTE = ("Fuente: PCM – decretos supremos (gob.pe), Open-Meteo (lluvia diaria en la capital)<br>"
          "e INEI (ubigeos). Temporadas ene–may 2022-2025. Elaboración propia.")
COLORES_TEMPORADA = ["#7fb6dc", "#4c94c8", "#2468a9", "#083b72"]   # un azul de claro a oscuro: más oscuro = más reciente
ORDEN_TEMPORADA = ["2022", "2023", "2024", "2025"]


def dar_formato(fig, y_fuente=-0.15):
    fig.update_layout(template="plotly_white", title_font_size=18,
                      margin=dict(l=10, r=10, t=130, b=120))
    fig.add_annotation(text=FUENTE, showarrow=False, xref="paper", yref="paper",
                       x=0, y=y_fuente, xanchor="left", yanchor="top", align="left",
                       font=dict(size=11, color="gray"))
    return fig

## 2. Gráfico de barras: ¿dónde se declara más emergencia?

Sumamos las declaratorias de las cuatro temporadas por departamento y nos quedamos con los 10 primeros. Hay un **empate en el 10.º lugar** (Cajamarca, Junín y San Martín, con 14 cada uno), así que usamos `keep="all"` para no dejar a ninguno fuera por el orden alfabético. Por eso el gráfico muestra 12 departamentos.

In [5]:
total_depto = df.groupby("departamento", as_index=False)["declaratorias"].sum()
top = total_depto.nlargest(10, "declaratorias", keep="all")
# Puesto en el ranking; los empates comparten puesto (por eso hay tres en el 10.º lugar)
top["puesto"] = top["declaratorias"].rank(method="min", ascending=False).astype(int)
top = top.set_index("puesto")
top

,departamento,declaratorias
puesto,,
1,Ayacucho,21
2,Lima,20
3,Arequipa,19
3,Áncash,19
5,Cusco,18
5,Huancavelica,18
5,Huánuco,18
8,Amazonas,16
9,Apurímac,15


In [6]:
barras = df[df["departamento"].isin(top["departamento"])].copy()
barras["temporada"] = barras["temporada"].astype(str)   # texto: un color por temporada

fig = px.bar(
    barras, x="declaratorias", y="departamento", color="temporada", orientation="h",
    color_discrete_sequence=COLORES_TEMPORADA,
    category_orders={"temporada": ORDEN_TEMPORADA},
    labels={"declaratorias": "Declaratorias de emergencia por lluvias (acumuladas)",
            "departamento": "", "temporada": "Temporada"},
    title="Ayacucho y Lima encabezan las declaratorias<br>de emergencia por lluvias",
    height=680,
)
fig.update_layout(title_subtitle_text="Declaratorias acumuladas 2022-2025 (empate en el 10.º lugar)")
fig.update_yaxes(categoryorder="total ascending",   # el mayor arriba
                 ticklabelstandoff=10)              # espacio entre el nombre y la barra
fig.update_traces(marker_line_color="white", marker_line_width=1.5)   # separa las temporadas dentro de cada barra
dar_formato(fig, y_fuente=-0.17)
fig.update_layout(margin_t=170)
fig.show()

**Verificación de valores extremos:** el departamento de arriba del gráfico debe ser el máximo de la tabla. Ayacucho suma 1 + 5 + 7 + 8 = 21 declaratorias, y al pasar el mouse por cada tramo de su barra aparecen esos mismos valores.

In [7]:
print("Máximo de la tabla:")
print(total_depto.loc[total_depto["declaratorias"].idxmax()])
print()
print(df.loc[df["departamento"] == "Ayacucho", ["temporada", "declaratorias"]].to_string(index=False))

Máximo de la tabla:
departamento     Ayacucho
declaratorias          21
Name: 3, dtype: object

 temporada  declaratorias
      2022              1
      2023              5
      2024              7
      2025              8


**Lectura:** los primeros lugares no son la selva, donde más llueve, sino departamentos andinos y de la costa (Ayacucho, Lima, Áncash, Arequipa). La barra de 2022 casi no se ve porque esa temporada tiene solo 3 declaratorias.

## 3. Gráfico de dispersión: ¿más lluvia, más declaratorias?

Si el Estado declarara emergencia donde más llueve, los puntos deberían subir de izquierda a derecha. Hacemos un panel por temporada (`facet_col`) para comparar los cuatro años. Al pasar el mouse sobre cada punto aparece el departamento (`hover_name`).

In [8]:
disp = df.copy()
disp["temporada"] = disp["temporada"].astype(str)

fig = px.scatter(
    disp, x="lluvia_total_mm", y="declaratorias",
    hover_name="departamento",
    hover_data={"capital": True, "dias_lluvia_fuerte": True, "prorrogas": True, "temporada": False},
    color="temporada", color_discrete_sequence=COLORES_TEMPORADA,
    category_orders={"temporada": ORDEN_TEMPORADA},
    facet_col="temporada", facet_col_wrap=2,
    facet_row_spacing=0.14, facet_col_spacing=0.08,
    labels={"lluvia_total_mm": "Lluvia acumulada en la capital (mm)",
            "declaratorias": "Declaratorias",
            "dias_lluvia_fuerte": "Días con 20 mm o más",
            "prorrogas": "Prórrogas", "capital": "Capital", "temporada": "Temporada"},
    title="Más lluvia no significa más declaratorias",
    height=850,
)
fig.update_layout(title_subtitle_text=("Loreto es el departamento más lluvioso todas las temporadas;<br>"
                                       "Lima, con menos de 60 mm, acumula 20 declaratorias"))


# Títulos de panel en negrita y alineados a la izquierda, para que no parezcan títulos de eje
def titulo_panel(anotacion):
    eje = next(e for e in fig.select_xaxes() if abs(sum(e.domain) / 2 - anotacion.x) < 0.01)
    anotacion.update(text=f"<b>{anotacion.text.replace('=', ' ')}</b>",
                     x=eje.domain[0], xanchor="left", font_size=14)


fig.for_each_annotation(titulo_panel)

# Números en los cuatro paneles y un solo título por eje (todos comparten la misma escala)
fig.update_xaxes(showticklabels=True, title_text="")
fig.update_yaxes(showticklabels=True, title_text="")
fig.add_annotation(text="Lluvia acumulada en la capital del departamento, enero–mayo (mm)",
                   x=0.5, y=-0.08, xref="paper", yref="paper", yanchor="top",
                   showarrow=False, font_size=14)
fig.add_annotation(text="Declaratorias de emergencia por lluvias",
                   x=-0.08, y=0.5, xref="paper", yref="paper", textangle=-90,
                   showarrow=False, font_size=14)

fig.update_traces(marker=dict(size=10, line=dict(width=1, color="white")))
fig.update_layout(showlegend=False)
dar_formato(fig, y_fuente=-0.15)
fig.update_layout(margin=dict(l=90, t=175, b=130))
fig.show()

**Verificación de valores extremos:** el punto más a la derecha (Loreto 2025) y el punto más alto (Cusco 2025) deben coincidir con los máximos de la tabla. También revisamos los dos casos que menciona el subtítulo.

In [9]:
columnas = ["temporada", "departamento", "lluvia_total_mm", "declaratorias"]
print("Más lluvia:")
print(df.loc[[df["lluvia_total_mm"].idxmax()], columnas].to_string(index=False))
print()
print("Más declaratorias en una temporada:")
print(df.loc[[df["declaratorias"].idxmax()], columnas].to_string(index=False))
print()
print("Lima y Loreto:")
print(df.loc[df["departamento"].isin(["Lima", "Loreto"]), columnas].to_string(index=False))

Más lluvia:
 temporada departamento  lluvia_total_mm  declaratorias
      2025       Loreto           2019.9              8

Más declaratorias en una temporada:
 temporada departamento  lluvia_total_mm  declaratorias
      2025        Cusco            801.3              9

Lima y Loreto:
 temporada departamento  lluvia_total_mm  declaratorias
      2022         Lima             39.2              0
      2022       Loreto           1581.8              0
      2023         Lima             55.5              6
      2023       Loreto           1604.6              1
      2024         Lima             17.3              6
      2024       Loreto           1361.8              3
      2025         Lima             54.0              8
      2025       Loreto           2019.9              8


Para medir la relación con un número, calculamos la **correlación de Pearson** entre lluvia y declaratorias en cada temporada. Va de -1 a 1:

- **1**: relación positiva perfecta. Los departamentos con más lluvia siempre tendrían más declaratorias, y los puntos formarían una línea recta que sube.
- **-1**: relación negativa perfecta. A más lluvia, siempre menos declaratorias.
- **0**: no hay relación lineal entre las dos variables.

Como referencia aproximada, una correlación menor a 0.3 en valor absoluto (sin importar el signo) se considera **débil**, entre 0.3 y 0.7 **moderada**, y mayor a 0.7 **fuerte**.

In [10]:
correlacion = (
    df.groupby("temporada")[["lluvia_total_mm", "declaratorias"]]
    .apply(lambda g: g["lluvia_total_mm"].corr(g["declaratorias"]))
    .round(2)
    .rename("correlacion_lluvia_declaratorias")
)
correlacion

temporada
2022   -0.17
2023   -0.22
2024    0.19
2025    0.20
Name: correlacion_lluvia_declaratorias, dtype: float64

**Lectura:** en ninguna temporada la correlación supera 0.22 en valor absoluto, así que la relación es débil en los cuatro años. En 2022 y 2023 incluso es negativa. Loreto, Ucayali y Madre de Dios, los más lluviosos, suman entre 6 y 12 declaratorias en cuatro temporadas, menos que Ayacucho (21); Lima, casi sin lluvia en su capital, tiene 6 a 8 por temporada desde 2023. La lluvia medida en la capital de cada departamento, por sí sola, no explica dónde se declara emergencia.

In [11]:
# Exportar el gráfico de dispersión a HTML (se abre con doble clic y sigue siendo interactivo)
fig.write_html("grafico_lluvia_vs_declaratorias.html")
print("Guardado: grafico_lluvia_vs_declaratorias.html")

Guardado: grafico_lluvia_vs_declaratorias.html


## 4. Gráfico de líneas: ¿cómo cambió entre temporadas?

Comparamos la evolución de la lluvia y de los decretos. Para la lluvia usamos el **promedio** de las 25 capitales, que es más fácil de leer que la suma. Cada indicador va en su propio panel con su propia escala (`facet_col` y `matches=None`, Sesión 6).

In [12]:
por_temporada = df.groupby("temporada", as_index=False).agg(
    lluvia_promedio_mm=("lluvia_total_mm", "mean"),
    declaratorias=("declaratorias", "sum"),
    prorrogas=("prorrogas", "sum"),
)
por_temporada["lluvia_promedio_mm"] = por_temporada["lluvia_promedio_mm"].round(1)
por_temporada

,temporada,lluvia_promedio_mm,declaratorias,prorrogas
0,2022,532.6,3,0
1,2023,643.7,72,6
2,2024,525.5,99,68
3,2025,708.4,157,71


In [13]:
nombres = {"lluvia_promedio_mm": "Lluvia promedio (mm)",
           "declaratorias": "Declaratorias", "prorrogas": "Prórrogas"}
largo = por_temporada.melt(id_vars="temporada", var_name="indicador", value_name="valor")
largo["indicador"] = largo["indicador"].map(nombres)

fig = px.line(
    largo, x="temporada", y="valor", facet_col="indicador", markers=True, text="valor",
    category_orders={"indicador": list(nombres.values())},
    labels={"temporada": "Temporada", "valor": "", "indicador": ""},
    title="Las declaratorias crecieron cada temporada,<br>aunque la lluvia subió y bajó",
    height=660,
)
fig.update_layout(title_subtitle_text=("De 3 declaratorias en 2022 a 157 en 2025;<br>"
                                       "la lluvia promedio fue parecida en 2022 y 2024"))
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1], font_size=14))
fig.update_yaxes(matches=None, showticklabels=True, rangemode="tozero")
fig.update_xaxes(dtick=1, range=[2021.5, 2025.5])   # margen para que no se corten las etiquetas
fig.update_traces(textposition="top center", texttemplate="%{text}", line_color="#2468a9")
dar_formato(fig, y_fuente=-0.25)
fig.update_layout(margin_t=220)   # título y subtítulo de 2 líneas cada uno, sobre los paneles
fig.show()

**Verificación de valores extremos:** los puntos de cada panel deben coincidir con la tabla `por_temporada`. El mínimo de declaratorias es 2022 (3) y el máximo 2025 (157). En lluvia, 2025 es la temporada más lluviosa (708.4 mm en promedio) y 2024 la más seca (525.5 mm).

In [14]:
tabla = por_temporada.set_index("temporada")
for columna in ["lluvia_promedio_mm", "declaratorias", "prorrogas"]:
    print(f"{columna}: mínimo {tabla[columna].min()} ({tabla[columna].idxmin()}), "
          f"máximo {tabla[columna].max()} ({tabla[columna].idxmax()})")

lluvia_promedio_mm: mínimo 525.5 (2024), máximo 708.4 (2025)
declaratorias: mínimo 3 (2022), máximo 157 (2025)
prorrogas: mínimo 0 (2022), máximo 71 (2025)


**Lectura:** 2024 llovió menos que 2023 (525.5 vs. 643.7 mm en promedio), pero hubo más declaratorias (99 vs. 72) y muchas más prórrogas (68 vs. 6). Las declaratorias crecen cada año aunque la lluvia no lo haga, lo que sugiere que también cuentan otros factores, como la vulnerabilidad de cada zona o cambios en cómo se usa la declaratoria. Las prórrogas también crecen: se mantienen emergencias declaradas en temporadas anteriores.

## 5. Hallazgos de la visualización

1. **No, el Estado no declara más emergencia donde más llueve.** La correlación entre lluvia y declaratorias está entre -0.22 y 0.20 en las cuatro temporadas. Loreto es el departamento más lluvioso todos los años y acumula 12 declaratorias; Lima, con menos de 60 mm por temporada en su capital, acumula 20.
2. **Las declaratorias se concentran en los Andes y la costa** (Ayacucho, Lima, Áncash, Arequipa), donde las lluvias causan huaicos y desbordes aunque llueva menos que en la selva.
3. **La lluvia siguió explicando poco en todas las temporadas, pero la cantidad de decretos creció cada año:** las declaratorias pasaron de 72 en 2023 a 157 en 2025 y las prórrogas de 6 a 71, mientras la lluvia promedio subió y bajó.

### Limitaciones

- **2022 tiene solo 3 declaratorias**, todas de un mismo decreto. Lo verificamos con los cuatro grupos que trabajaron esa temporada y coincide, pero con tan pocos casos esa temporada aporta poco a la comparación. Además, todos los datos dependen de lo que devuelve el buscador de gob.pe: si un decreto no aparece en la búsqueda, o su página no menciona la lluvia, no se cuenta.
- **La lluvia se mide en un solo punto** (la capital del departamento). En departamentos grandes o con costa, sierra y selva, como Lima, no representa lo que llovió en los distritos afectados.
- **Contamos declaratorias por departamento**, no por distrito: un decreto que incluye 2 distritos pesa lo mismo que uno que incluye 200.
- **Unimos datos de cuatro grupos distintos.** Cada temporada la scrapeó un grupo diferente y, aunque todos siguieron el mismo enunciado, cada uno tomó sus propias decisiones. Al comparar nuestra tabla con las de otros grupos que trabajaron la misma temporada encontramos tres diferencias:
  - **Lima 2023:** el grupo 2, de donde tomamos esa temporada, cuenta 6 declaratorias; los grupos 6 y 10 cuentan 7, porque incluyen el Decreto Supremo N.° 036-2023-PCM, cuya página en gob.pe no menciona la lluvia. Mantuvimos los datos del grupo 2.
  - **Ucayali 2024:** los grupos 7 y 11 cuentan 4 declaratorias y nosotros 3. La diferencia es el Decreto Supremo N.° 049-2024-PCM, que menciona la *provincia* de Ucayali del departamento de Loreto, no el departamento de Ucayali.
  - **Capital de Lima:** algunos grupos midieron la lluvia en Huacho y otros en la ciudad de Lima. Nuestra tabla usa Lima en las cuatro temporadas, así que la serie es consistente.